# 02_nettoyage_description_stations  

- **Structure** : dimensions et types, changement de type, colonnes supprimées ou renommées  
- **Qualité** : valeurs manquantes, doublons, valeurs aberrantes, continuuité en série, sens métier de chaque colonne ?  
- **Traçabilité** : colonens supprimées et écrire pourquoi dans cellule Markdown + exporter en fichier propre dans /inter


### PLAN DU NOTEBOOK

**Objectif** : à partir des données brutes téléchargées par `01_collectAPI_stations`, contrôler leur qualité et analyser (structure, qualité, traçabilité).

- Source : API Hub'Eau « Stations » (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Le projet se fait en quatres notebooks :
    1. `01_collectAPI_stations` et `01_collectAPI_chroniques`: liste des stations de la zone (endpoint `/stations`) et téléchargement des mesures de chaque station (endpoint `/chroniques`);
    2. **`02_nettoyage_description_stations.ipynb`(ce notebook)** et `02_nettoyage_description_chroniques` : contrôle qualité, préparation + enregistrement dans le dossier `/inter`.
    3. `03_analyse-EDA_stations` et `03_analyse-EDA_chroniques` : analyse univarié et bivarié, graphique de visualisation et analyse temporelle, début des analyse inférentielle. 
    4. `04_fusion-EDA_stations-chroniques.ipynb` : fusion des tables, analyse inférentielle et conclusion.

**Entrées** (aucun appel à l'API dans ce notebook)
- `stations_extrait_le_AAAA-MM-JJ.json` : produit par `01_collectAPI_stations`

**Étapes**  
0. Imports des bibliothèques  
1. 
...
...
...

**Sorties**
- `data/inter/stations.csv`

In [ ]:
# 0. Imports des bibliothèques

# Bibliothèque standard
import json                         # lecture du fichier brut
from pathlib import Path            # chemins de fichiers

# Bibliothèques externes
import pandas as pd                 # mise en DataFrame et contrôle qualité

# --- Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTER = PROJECT_ROOT / "data" / "inter"
DATA_INTER.mkdir(parents=True, exist_ok=True)


#### 1. Chargement des données et structure :
- 1a. Chargement du fichier brut + mise en dataframe + aperçu
- 1b. Dimensions et types
- 1c. Listing des colonnes
- changement de type
- colonnes supprimées
- colonnes renommées 

   - 6a. Création et aperçu du DataFrame
   - 6b. Contrôle qualité : types, valeurs manquantes, doublons
   - 6c. Conversion des types
   - 6d. Sélection des stations avec mesures

In [9]:
# 1a. Chargement du fichier brut + mise en dataframe + aperçu

# Fichier brut le plus récent produit par 01_collectAPI_stations
fichier_brut = sorted(DATA_RAW.glob("stations_extrait_le_*.json"))[-1]

with open(fichier_brut, encoding="utf-8") as f:
    contenu_brut = json.load(f)

print("Fichier lu         :", fichier_brut.name)
print("Date d'extraction  :", contenu_brut["date_extraction"])

df_stations = pd.DataFrame(contenu_brut["data"])   # les stations sont dans la clé "data"

assert len(df_stations) == contenu_brut["nb_resultats"], "⚠️ Nombre de stations incohérent"
print("Dimensions (lignes, colonnes) :", df_stations.shape)
df_stations.head()


Fichier lu         : stations_extrait_le_2026-11-16.json
Date d'extraction  : 2026-11-16T15:52:14
Dimensions (lignes, colonnes) : (176, 22)


,code_bss,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,x,y,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,07702X0081/1,http://services.ades.eaufrance.fr/pointeau/077...,NaN,NaN,26325,Saint-Rambert-d'Albon,4.837284,45.289166,None,[],...,164.0,0,26,Drôme,NaN,27.25,None,None,[],Wed Jun 25 20:01:28 CEST 2025
1,07477X0032/3821,http://services.ades.eaufrance.fr/pointeau/074...,NaN,NaN,38130,La Côte-Saint-André,5.274229,45.373098,None,[],...,349.0,0,38,Isère,NaN,20.50,None,None,[],Wed Jun 25 20:01:28 CEST 2025
2,08422X0190/F1,http://services.ades.eaufrance.fr/pointeau/084...,1999-12-17,2026-11-13,26166,Loriol-sur-Drôme,4.791298,44.756364,"[529AA00, 712BM01]",[http://reseau.eaufrance.fr/geotraitements/bdl...,...,95.0,9157,26,Drôme,PIEZOMETRE - AUTOROUTE (LORIOL SUR DROME - BRG...,11.50,[DG337],[Alluvions de la Drôme],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Sun Jul 14 13:00:02 CEST 2024
3,07704X0083/F1,http://services.ades.eaufrance.fr/pointeau/077...,NaN,NaN,26155,Lapeyrouse-Mornay,4.961601,45.322710,None,[],...,240.0,0,26,Drôme,NaN,46.00,None,None,[],Wed Jun 25 20:01:28 CEST 2025
4,07948X0047/F1,http://services.ades.eaufrance.fr/pointeau/079...,2009-06-12,2026-11-13,26281,Romans-sur-Isère,5.025114,45.067221,[521AU00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,210.0,6136,26,Drôme,FORAGE - MAUPAS (ROMANS SUR ISERE - 26),220.00,None,None,[],Fri Jun 28 07:31:38 CEST 2024


In [8]:
# 1b. Dimensions et types
# Dimensions du df
df_stations.shape

# Types des colonnes
df_stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   code_bss                  176 non-null    str           
 1   urn_bss                   176 non-null    str           
 2   date_debut_mesure         60 non-null     datetime64[us]
 3   date_fin_mesure           60 non-null     datetime64[us]
 4   code_commune_insee        176 non-null    str           
 5   nom_commune               176 non-null    str           
 6   x                         176 non-null    float64       
 7   y                         176 non-null    float64       
 8   codes_bdlisa              85 non-null     object        
 9   urns_bdlisa               176 non-null    object        
 10  geometry                  176 non-null    object        
 11  bss_id                    176 non-null    str           
 12  altitude_station          176 non

In [11]:
# 1c. Listing des colonnes : type, catégorie, valeurs différentes, manquantes et choix possibles
    # - Colonne numérique   : type nombre (int, float), résumée par son minimum et son maximum
    # - Colonne catégorielle : tout le reste (texte, dates, booléens...)
    # Les choix possibles ne sont listés que si la colonne a peu de valeurs différentes (≤ MAX_MODALITES)
    # Le résultat est affiché ET enregistré dans un fichier texte (data/inter/)

MAX_MODALITES = 20          # au-delà, on n'énumère pas les valeurs (identifiants, coordonnées...)
UNITE = "stations"          # unité comptée en face de chaque choix
NOM_FICHIER = "stations"    # préfixe du fichier texte produit

# En-tête du fichier : origine des données (traçabilité)
lignes = [f"Contenu des colonnes du fichier {fichier_brut.name}",
          f"Date d'extraction : {contenu_brut['date_extraction']}",
          f"{len(df_stations)} {UNITE}, {df_stations.shape[1]} colonnes",
          ""]

for colonne in df_stations.columns:
    serie = df_stations[colonne]

    # Certaines colonnes de l'API contiennent des dict/listes (ex. geometry) : on les passe en texte
    if serie.map(lambda v: isinstance(v, (dict, list))).any():
        serie = serie.map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)

    nb_valeurs = serie.nunique(dropna=True)
    nb_manquants = serie.isna().sum()
    est_numerique = pd.api.types.is_numeric_dtype(serie) and not pd.api.types.is_bool_dtype(serie)
    categorie = "numérique" if est_numerique else "catégorielle"

    # Ligne 1 : nom de la colonne
    lignes.append(f"--- {colonne} ---")
    # Ligne 2 : type, catégorie, nombre de valeurs différentes, nombre de manquantes
    lignes.append(f"    {serie.dtype}, {categorie}, {nb_valeurs} valeur(s) différente(s), {nb_manquants} manquante(s)")

    # Lignes suivantes : choix possibles avec leur effectif
    if nb_valeurs <= MAX_MODALITES:
        for valeur, nb in serie.value_counts(dropna=False).items():
            lignes.append(f"      - {valeur!s:<45} {nb:>6} {UNITE}")
    elif est_numerique:
        lignes.append(f"      de {serie.min()} à {serie.max()}")
    else:
        lignes.append(f"      (trop de valeurs pour être listées) exemple : {serie.dropna().iloc[0]}")
    lignes.append("")

texte = "\n".join(lignes)
print(texte)

# Enregistrement dans un fichier texte (data/inter/)
fichier_contenu = DATA_INTER / f"{NOM_FICHIER}_contenu_colonnes.txt"
fichier_contenu.write_text(texte, encoding="utf-8")
print(f"✅ Contenu des colonnes enregistré dans : {fichier_contenu}")


Contenu des colonnes du fichier stations_extrait_le_2026-11-16.json
Date d'extraction : 2026-11-16T15:52:14
176 stations, 22 colonnes

--- code_bss ---
    str, catégorielle, 176 valeur(s) différente(s), 0 manquante(s)
      (trop de valeurs pour être listées) exemple : 07702X0081/1

--- urn_bss ---
    str, catégorielle, 176 valeur(s) différente(s), 0 manquante(s)
      (trop de valeurs pour être listées) exemple : http://services.ades.eaufrance.fr/pointeau/07702X0081/1

--- date_debut_mesure ---
    str, catégorielle, 46 valeur(s) différente(s), 116 manquante(s)
      (trop de valeurs pour être listées) exemple : 1999-12-17

--- date_fin_mesure ---
    str, catégorielle, 21 valeur(s) différente(s), 116 manquante(s)
      (trop de valeurs pour être listées) exemple : 2026-11-13

--- code_commune_insee ---
    str, catégorielle, 82 valeur(s) différente(s), 0 manquante(s)
      (trop de valeurs pour être listées) exemple : 26325

--- nom_commune ---
    str, catégorielle, 82 valeur(s) d

In [3]:
# 6b. Contrôle qualité : types, valeurs manquantes, doublons

# Valeurs manquantes par colonne (uniquement celles qui en ont)
manquants = df_stations.isna().sum()
print("\nValeurs manquantes :\n", manquants[manquants > 0].sort_values(ascending=False))
print("\n")
      
# Types des colonnes
df_stations.info()

# Doublons sur l'identifiant de station
print("\nDoublons sur code_bss :", df_stations["code_bss"].duplicated().sum())


Valeurs manquantes :
 date_debut_mesure           116
date_fin_mesure             116
noms_masse_eau_edl          111
codes_masse_eau_edl         111
codes_bdlisa                 91
libelle_pe                   41
profondeur_investigation     31
dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   code_bss                  176 non-null    str    
 1   urn_bss                   176 non-null    str    
 2   date_debut_mesure         60 non-null     str    
 3   date_fin_mesure           60 non-null     str    
 4   code_commune_insee        176 non-null    str    
 5   nom_commune               176 non-null    str    
 6   x                         176 non-null    float64
 7   y                         176 non-null    float64
 8   codes_bdlisa              85 non-null     object 
 9   urns_bdlisa               176 non-

**Constat :** 116 stations sur 176 ont `nb_mesures_piezo = 0` (valeur renseignée mais égale à zéro).
Elles n'ont donc pas de dates de mesure, ce qui explique les 116 valeurs manquantes sur
`date_debut_mesure` et `date_fin_mesure`. Aucun doublon sur `code_bss`.

In [4]:
# 6c. Conversion des types

df_stations["altitude_station"] = pd.to_numeric(df_stations["altitude_station"], errors="coerce")
df_stations["date_debut_mesure"] = pd.to_datetime(df_stations["date_debut_mesure"])
df_stations["date_fin_mesure"] = pd.to_datetime(df_stations["date_fin_mesure"])

df_stations[["altitude_station", "date_debut_mesure", "date_fin_mesure"]].dtypes


altitude_station            float64
date_debut_mesure    datetime64[us]
date_fin_mesure      datetime64[us]
dtype: object

In [5]:
# 6d. Sélection des stations ayant au moins une mesure de niveau d'eau

df_stations_mesurees = df_stations[df_stations["nb_mesures_piezo"] > 0].copy()

print(f"{len(df_stations_mesurees)} stations avec mesures sur {len(df_stations)}")
df_stations_mesurees[["code_bss", "nom_commune", "nom_departement",
                      "date_debut_mesure", "date_fin_mesure", "nb_mesures_piezo"]].head(10)


60 stations avec mesures sur 176


,code_bss,nom_commune,nom_departement,date_debut_mesure,date_fin_mesure,nb_mesures_piezo
2,08422X0190/F1,Loriol-sur-Drôme,Drôme,1999-12-17,2026-11-13,9157
4,07948X0047/F1,Romans-sur-Isère,Drôme,2009-06-12,2026-11-13,6136
5,08664X0069/F1,Soyans,Drôme,2008-12-03,2026-11-11,6526
6,07703X0064/P1,Anneyron,Drôme,2009-06-09,2026-04-20,6155
8,08187X0238/P1,Beaumont-lès-Valence,Drôme,2009-06-26,2026-11-13,6343
9,07955X0099/P1,Chatuzange-le-Goubet,Drôme,2005-11-24,2026-11-13,7450
12,08184X0084/PZ1,Valence,Drôme,1982-02-04,2026-10-21,9637
14,08422X0191/F2,Livron-sur-Drôme,Drôme,1999-12-17,2026-11-12,8469
17,08424X0006/F2,Eurre,Drôme,1995-06-29,2026-11-13,8900
20,08426X0133/P2,Saulce-sur-Rhône,Drôme,2009-06-17,2026-11-13,6249


In [6]:
# 7. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"stations_{date_extraction:%Y-%m-%d}.json"

# On enregistre les données ET le contexte de l'extraction (traçabilité)
contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_STATIONS,
    "params": params_stations,
    "nb_resultats": len(stations),
    "data": stations,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False, indent=2)

print(f"✅ {len(stations)} stations sauvegardées dans : {fichier_brut}")


NameError: name 'datetime' is not defined

In [ ]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de stations :", len(relu["data"]))
assert len(relu["data"]) == len(stations), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

NameError: name 'fichier_brut' is not defined

In [ ]:
# 8. Sauvegarde des données traitées (data/processed/) : entrée du notebook 01_collectAPI_chroniques

fichier_stations = DATA_PROCESSED / "stations_mesurees.csv"
df_stations_mesurees.to_csv(fichier_stations, index=False)

print(f"✅ {len(df_stations_mesurees)} stations avec mesures sauvegardées dans : {fichier_stations}")

NameError: name 'DATA_PROCESSED' is not defined

In [ ]:
# 8b. Sauvegarde de la totalité des stations, avec un indicateur de présence de mesures

df_stations_total = df_stations.copy()
df_stations_total["avec_mesures"] = df_stations_total["nb_mesures_piezo"] > 0   # True si au moins 1 mesure

fichier_stations_total = DATA_PROCESSED / "stations_total.csv"
df_stations_total.to_csv(fichier_stations_total, index=False)

print(f"✅ {len(df_stations_total)} stations sauvegardées dans : {fichier_stations_total}")
print(df_stations_total["avec_mesures"].value_counts())

NameError: name 'df_stations' is not defined

#### 9. Export GeoJSON

Conversion des stations en **GeoDataFrame** (geopandas) puis export au format **GeoJSON**,
lisible directement par les outils cartographiques (QGIS, geopandas, folium, geojson.io…).

**2 fichiers GeoJSON sont produits :**

| Fichier | Contenu | Usage |
|---|---|---|
| `stations_total.geojson` | Les **176 stations**, avec la colonne `avec_mesures` (True / False) | Carte de toutes les stations, avec un **filtre** ou une couleur selon `avec_mesures` |
| `stations_mesurees.geojson` | Uniquement les **60 stations avec mesures** | Carte des seules stations exploitables pour les chroniques |

- Le second fichier est obtenu en **filtrant le premier** sur `avec_mesures` : les deux restent cohérents.
- La géométrie (un point par station) est construite à partir des colonnes `x` (longitude) et `y` (latitude).
- `crs="EPSG:4326"` : système de coordonnées WGS84 (degrés), celui utilisé par l'API Hub'Eau.
- Seules les colonnes simples sont exportées : les colonnes contenant des listes ou des dictionnaires
  (`geometry` d'origine, `urns_*`, `codes_*`…) restent disponibles dans le fichier brut JSON.

In [ ]:
# 9. Export GeoJSON (data/processed/) : totalité des stations + stations avec mesures

COLONNES_GEO = [
    "code_bss", "bss_id", "nom_commune", "code_commune_insee",
    "code_departement", "nom_departement",
    "altitude_station", "profondeur_investigation", "nb_mesures_piezo",
    "date_debut_mesure", "date_fin_mesure",
    "x", "y",
    "avec_mesures",                     # True / False : filtre pour la cartographie
]

# 9a. Toutes les stations (176), avec la colonne avec_mesures
gdf_stations_total = gpd.GeoDataFrame(
    df_stations_total[COLONNES_GEO],
    geometry=gpd.points_from_xy(df_stations_total["x"], df_stations_total["y"]),
    crs="EPSG:4326",                    # WGS84 : longitude / latitude en degrés
)

# 9b. Uniquement les stations avec mesures (60) : filtre du fichier total
gdf_stations_mesurees = gdf_stations_total[gdf_stations_total["avec_mesures"]].copy()

# Export des 2 fichiers
fichier_geojson_total = DATA_PROCESSED / "stations_total.geojson"
fichier_geojson_mesurees = DATA_PROCESSED / "stations_mesurees.geojson"

gdf_stations_total.to_file(fichier_geojson_total, driver="GeoJSON")
gdf_stations_mesurees.to_file(fichier_geojson_mesurees, driver="GeoJSON")

print(f"✅ {len(gdf_stations_total)} stations (toutes) exportées dans    : {fichier_geojson_total}")
print(f"✅ {len(gdf_stations_mesurees)} stations (avec mesures) exportées dans : {fichier_geojson_mesurees}")

NameError: name 'gpd' is not defined

In [ ]:
# Vérification : relecture des 2 fichiers GeoJSON

for fichier, gdf_attendu in [(fichier_geojson_total, gdf_stations_total),
                             (fichier_geojson_mesurees, gdf_stations_mesurees)]:
    gdf_relu = gpd.read_file(fichier)
    print(f"--- {fichier.name}")
    print("  Nombre de stations     :", len(gdf_relu))
    print("  Système de coordonnées :", gdf_relu.crs)
    print("  Type de géométrie      :", gdf_relu.geom_type.unique().tolist())
    print("  avec_mesures           :", gdf_relu["avec_mesures"].value_counts().to_dict())
    assert len(gdf_relu) == len(gdf_attendu), f"⚠️ {fichier.name} ne correspond pas aux données exportées"

NameError: name 'fichier_geojson_total' is not defined